# Kostic et al. 2023 Metrics 

## Functions for implementing and analysing metrics

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from kooplearn._linalg import eigh_rank_reveal, spd_neg_pow, weighted_norm
from kooplearn.datasets import compute_prinz_potential_eig, make_prinz_potential
from kooplearn.kernel import KernelRidge


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


def metric_distortion(psi, C):
    r"""Empirical metric distortion :math:`\widehat\eta_i = \|\widehat\psi_i\|_{\mathcal H} /
    \sqrt{\langle \widehat C \widehat\psi_i, \widehat\psi_i\rangle}`.

    Parameters
    ----------
    psi : ndarray, shape (n,) or (n, k)
        Eigenfunction(s) evaluated at the *training* points. If 2D, each
        column is treated as a separate eigenfunction (see `weighted_norm`).
    C : ndarray, shape (n, n)
        Empirical (kernel-based) covariance, i.e. ``model.kernel_X / n_samples``.
    """
    psi = np.asarray(psi)
    n = C.shape[0]

    # ||psi||_H via the reproducing property: needs the *inverse* Gram, since
    # C = K_X / n is the Gram-based covariance, not the RKHS metric itself.
    C_inv = spd_neg_pow(C * n, exponent=-1.0)  # i.e. K_X^{-1}
    rkhs_norm = weighted_norm(psi, M=C_inv)

    # <C psi, psi> = (1/n)||psi(X)||_2^2, i.e. weighted_norm with M=None, squared, over n
    empirical_norm = weighted_norm(psi, M=None) / np.sqrt(n)

    with np.errstate(divide="ignore", invalid="ignore"):
        eta = rkhs_norm / empirical_norm
    eta = np.where(empirical_norm > 0, eta, np.nan)
    return eta if psi.ndim == 2 else float(eta)


def spectral_bias(eigenfunction, C, rho):
    r"""Empirical spectral bias :math:`\hat s_i = \widehat\eta_i \, \rho_{r+1}`."""
    eta = metric_distortion(eigenfunction, C)
    s_hat = eta * rho
    return float(s_hat), eta


def _top_sv(C, r):
    """(r+1)-st eigenvalue of a symmetric PSD matrix, via eigh_rank_reveal."""
    raw_vals, raw_vecs = np.linalg.eigh(np.asarray(C))
    _, top_vals, _ = eigh_rank_reveal(raw_vals, raw_vecs, rank=r + 1)
    if len(top_vals) <= r:
        return 0.0
    return float(top_vals[-1])


# --- truncation helpers ---
def pcr_truncation(C, r):
    r""":math:`\rho_{r+1}(\widehat G^{PCR}) = \sigma_{r+1}(\widehat C)`."""
    return _top_sv(C, r)


# kDMD uses the same (r+1)-st eigenvalue of the empirical covariance as PCR
kdmd_truncation = pcr_truncation


def rrr_truncation(C, T, r, cutoff=None):
    r""":math:`\rho_{r+1}(\widehat G^{RRR}) = \sigma_{r+1}(\widehat C^{-1/2}\widehat T)`."""
    C_inv_sqrt = spd_neg_pow(np.asarray(C), exponent=-0.5, cutoff=cutoff)
    A = C_inv_sqrt @ np.asarray(T)
    svals = np.linalg.svd(A, compute_uv=False)
    if r >= len(svals):
        return 0.0
    return float(svals[r])


# --- spectral gap (top-two magnitude eigenvalues) ---
def spectral_gap(eigenvalues):
    mags = np.sort(np.abs(eigenvalues))[::-1]
    return float(mags[0] - mags[1]) if len(mags) > 1 else np.nan


# --- spurious eigenvalues vs reference ---


def spurious_ref(est, ref, delta):
    dist = np.abs(est[:, None] - ref[None, :])
    return int(np.sum(dist.min(axis=1) > delta))


def spurious_residual(eigenvalues, psi_X_val, psi_Y_val, delta, relative=True):
    r"""Data-driven spurious-eigenpair check (see paper Appendix C, Remark 4).

    Flags eigenpairs that fail the empirical consistency check
    :math:`\hat\psi_i(y_j) \approx \hat\lambda_i \hat\psi_i(x_j)` on a
    held-out validation set, i.e. large :math:`\|(\widehat Z - \widehat S\widehat G)\hat\psi_i\|`.

    Parameters
    ----------
    eigenvalues : ndarray, shape (r,)
        Estimated eigenvalues, same order as columns of psi_X_val/psi_Y_val.
    psi_X_val : ndarray, shape (n_val, r)
        Eigenfunctions evaluated at validation inputs x_j.
    psi_Y_val : ndarray, shape (n_val, r)
        Same eigenfunctions evaluated at the corresponding outputs y_j = f(x_j).
    delta : float
        Threshold on the (optionally normalized) residual.
    relative : bool
        If True, normalize residual by ||psi_X_val||, giving a scale-free score.

    Returns
    -------
    n_spurious : int
    scores : ndarray, shape (r,)
    """
    eigenvalues = np.asarray(eigenvalues)
    n_val = psi_X_val.shape[0]

    resid = psi_Y_val - psi_X_val * eigenvalues[None, :]
    resid_norm = weighted_norm(resid) / np.sqrt(n_val)

    if relative:
        base_norm = weighted_norm(psi_X_val) / np.sqrt(n_val)
        scores = np.where(base_norm > 0, resid_norm / base_norm, np.nan)
    else:
        scores = resid_norm

    n_spurious = int(np.sum(scores > delta))
    return n_spurious, scores


# --- compilation function for analysing spectral metrics ---
def analyse_spectrum(modes_records, trials_records, out_prefix):
    modes_df = pd.DataFrame(modes_records).copy()
    trials_df = pd.DataFrame(trials_records).copy()

    summary = modes_df.groupby(
        ["kernel", "kind", "method", "eigenfunction_id"], as_index=False
    ).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        dist_mean=("metric_distortion", "mean"),
        trunc_mean=("truncation", "mean"),
        spurious_mean=("residual_spurious_score", "mean"),
        spurious_std=("residual_spurious_score", "std"),
    )
    # modes_df = modes_df.merge(
    #    trials_df[["trial", "spectral_gap"]].rename(columns={"spectral_gap": "gap"}),
    #    on="trial",
    #    how="left",
    # )
    corr_df = (
        modes_df.groupby(["kernel", "kind", "method"])
        .apply(lambda g: g["spectral_bias"].corr(g["gap"]))
        .rename("bias_gap_corr")
        .reset_index()
    )

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    modes_df.to_csv(f"{out_prefix}_metrics.csv", index=False)
    trials_df.to_csv(f"{out_prefix}_trials.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_corr.csv", index=False)

    fig1, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["gap"],
            s=20,
            alpha=0.7,
            label=f"{kernel}, {kind} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Spectral gap")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs Spectral gap")
    fig1.tight_layout()
    fig1.savefig(f"{out_prefix}_gap_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig1)

    fig2, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["residual_spurious_score"],
            s=20,
            alpha=0.7,
            label=f"{kernel}, {kind} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Residual spurious score")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs Residual spurious score")
    fig2.tight_layout()
    fig2.savefig(f"{out_prefix}_spurious_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig2)

    return modes_df, trials_df, summary, corr_df, fig1, fig2


## Overdamped Langevin Spectral Analysis

In [ ]:
x = np.linspace(-2, 2, 2048 + 1)
gamma = 1.0
sigma = 2.0
dt = 1e-4
subsample = 100
n_train = 2000
n_val = 500
n_trials = 10
n_show = 3
# training simulation:
n_steps_train = n_train * subsample
# validation simulation:
n_steps_val = n_val * subsample

data = make_prinz_potential(X0=0, n_steps=n_steps_train, gamma=gamma, sigma=sigma, random_state=0)
data = data.iloc[::subsample][:n_train]


# Validation set
n_val = 500
data_val = make_prinz_potential(
    X0=0, n_steps=n_steps_val, gamma=gamma, sigma=sigma, random_state=999
)  # different seed
data_val = data_val.iloc[::subsample][:n_val]

X_val = data_val.iloc[:-1].values
Y_val = data_val.iloc[1:].values


vals_ref = compute_prinz_potential_eig(gamma, sigma, dt, num_components=5)

trials_records = []
modes_records = []
for method, reduced_rank in zip(
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
):
    for trial in tqdm(range(n_trials), desc=method):
        model = KernelRidge(
            n_components=5,
            reduced_rank=reduced_rank,
            gamma=12.5,
            kernel="rbf",
            alpha=1e-6,
            random_state=trial,  # vary the seed across trials
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.05)

        # Evaluate the SAME fitted eigenfunctions at validation points (unsorted call, then reorder)
        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        # sanity check: eigenvalues from this call should match vals_hat up to the same permutation
        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            kdmd_truncation(C, fit_rank)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, fit_rank)
        )

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes
        trials_records.append(
            {
                "kind": "N/A",
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),  # reference against true spectrum σ(A)
                "spurious_residual_count": int(n_residual),  # data-driven, appendix-style
                "spectral_gap": gap,
                "rank": fit_rank,
            }
        )

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes
        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)
            modes_records.append(
                {
                    "kind": "N/A",
                    "kernel": "rbf",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": gap,
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )

modes_df, trials_df, summary, corr_df, fig1, fig2 = analyse_spectrum(
    modes_records, trials_records, out_prefix="langevin_full"
)
print(modes_df)
print(trials_df)
print(summary)
print(corr_df)


Reduced Rank: 100%|██████████| 10/10 [57:56<00:00, 347.66s/it]


    kind kernel                       method  trial  eigenfunction_id  \
0    N/A    rbf  Principal Components (kDMD)      0                 1   
1    N/A    rbf  Principal Components (kDMD)      0                 1   
2    N/A    rbf  Principal Components (kDMD)      0                 2   
3    N/A    rbf  Principal Components (kDMD)      0                 2   
4    N/A    rbf  Principal Components (kDMD)      0                 3   
..   ...    ...                          ...    ...               ...   
195  N/A    rbf                 Reduced Rank      9                 3   
196  N/A    rbf                 Reduced Rank      9                 4   
197  N/A    rbf                 Reduced Rank      9                 4   
198  N/A    rbf                 Reduced Rank      9                 5   
199  N/A    rbf                 Reduced Rank      9                 5   

     spectral_bias  metric_distortion  truncation  residual_spurious_score  \
0         0.581365          11.980713    0.04

## OU spectral analysis

In [15]:
import itertools
from math import factorial

import numpy as np
from numpy.polynomial.hermite_e import hermeval

x_grid = np.linspace(-4, 4, 1025)[:, None]  # for later plotting eigenfunctions, if wanted
gamma = 1.0
dt = 1e-4
subsample = 100
n_train = 2000
n_val = 500
n_trials = 10
n_show = 3
M = 10
n_components = 5
r = n_components  # target rank for the good/bad/ugly permutation construction
n_burn = 200
# if one trajectory: n_steps = (n_train + n_val + n_burn) * subsample


def hermite_prob(n, x):
    c = np.zeros(n + 1)
    c[n] = 1.0
    return hermeval(x, c)


def hermite_features(x, M):
    x = np.asarray(x).reshape(-1)
    Phi = np.zeros((x.shape[0], M))
    for j in range(M):
        Phi[:, j] = hermite_prob(j, x) / np.sqrt(factorial(j))
    return Phi


def kernel_permutation(kind, r, M):
    pi = np.arange(M)
    if kind == "good":
        return pi
    if kind == "bad":
        if 2 * r > M:
            raise ValueError(f"Need M >= 2*r for bad kernel, got M={M}, r={r}")
        pi2 = pi.copy()
        a, b = np.arange(r), np.arange(r, 2 * r)
        pi2[a], pi2[b] = b[::-1], a[::-1]
        return pi2
    if kind == "ugly":
        return pi[::-1]
    raise ValueError(kind)


def make_weights(kind, r, M):
    mu = np.exp(-np.arange(M))
    nu = 1.0 if kind == "good" else (1.0 / (r**2) if kind == "bad" else r**2)
    pi = kernel_permutation(kind, r, M)
    return mu[pi] ** (2.0 * nu)


def hermite_kernel(kind, r, M):
    w = make_weights(kind, r, M)

    def kernel(x, y):
        x = np.asarray(x).ravel()
        y = np.asarray(y).ravel()
        fx = np.array([hermite_prob(m, x[0]) / np.sqrt(factorial(m)) for m in range(M)])
        fy = np.array([hermite_prob(m, y[0]) / np.sqrt(factorial(m)) for m in range(M)])
        return float(np.sum(w * fx * fy))

    return kernel


def simulate_ou(n_steps, gamma, dt, random_state, x0=0.0):
    """AR(1) sampled from the exact OU transition at physical step size dt."""
    rng = np.random.default_rng(random_state)
    a = np.exp(-gamma * dt)
    b = np.sqrt(1.0 - a**2)
    x = np.empty(n_steps, dtype=float)
    x[0] = float(x0)
    for t in range(n_steps - 1):
        x[t + 1] = a * x[t] + b * rng.standard_normal()
    return pd.DataFrame({"x": x})


def compute_ou_eig(gamma, lag, num_components):
    n = np.arange(num_components)
    return np.exp(-n * gamma * lag)


n_steps_train = n_train * subsample
n_steps_val = n_val * subsample

data = simulate_ou(
    n_steps=n_steps_train,
    gamma=gamma,
    dt=dt,
    random_state=0,
    x0=0.0,
).iloc[::subsample][:n_train]

data_val = simulate_ou(
    n_steps=n_steps_val,
    gamma=gamma,
    dt=dt,
    random_state=999,
    x0=0.0,
)
data_val = data_val.iloc[::subsample].reset_index(drop=True)[:n_val]

X_val = data_val.iloc[:-1].values
Y_val = data_val.iloc[1:].values

lag = dt * subsample  # consistent with how `data` was actually generated
vals_ref = compute_ou_eig(gamma, lag, num_components=n_components)

trials_records = []
modes_records = []
for method, reduced_rank, kind in itertools.product(
    ["PCR", "Reduced Rank"], [False, True], ["good", "bad", "ugly"]
):
    for trial in tqdm(range(n_trials), desc=f"{method}/{kind}"):
        kernel = hermite_kernel(kind, r, M)
        model = KernelRidge(
            n_components=n_components,
            reduced_rank=reduced_rank,
            gamma=12.5,
            kernel=kernel,
            alpha=0,
            random_state=trial,
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.05)

        # Evaluate the SAME fitted eigenfunctions at validation points (unsorted call, then reorder)
        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        # sanity check: eigenvalues from this call should match vals_hat up to the same permutation
        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            rrr_truncation(C, T, fit_rank)
            if method == "Reduced Rank"
            else pcr_truncation(C, fit_rank)
        )
        trials_records.append(
            {
                "kind": kind,
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),  # reference against true spectrum σ(A)
                "spurious_residual_count": int(n_residual),  # data-driven, appendix-style
                "spectral_gap": gap,
                "rank": fit_rank,
            }
        )

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes
        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)
            modes_records.append(
                {
                    "kind": kind,
                    "kernel": "hermite",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": gap,
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )

modes_df, trials_df, summary, corr_df, fig1, fig2 = analyse_spectrum(
    modes_records, trials_records, "ou_full"
)
print(modes_df)
print(trials_df)
print(summary)
print(corr_df)


PCR/good:  10%|█         | 1/10 [9:04:56<81:44:24, 32696.11s/it]


KeyboardInterrupt: 

N.B. `model.dynamical_modes(data)` contains:

```python
[
    'U_', 
    'V_', 
    'X_fit_', 
    'feature_names_in_', 
    'gamma_', 
    'kernel_X_', 
    'kernel_YX_', 
    'kernel_Y_', 
    'n_features_in_', 
    'rank_', 
    'y_fit_'
]
```